In [1]:
import pandas as pd

import joblib

import optuna
from optuna.samplers import TPESampler

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score


optuna.logging.set_verbosity(optuna.logging.WARNING)

/workspaces/Northstar-Desk-Decision-Support-Tool-/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X_train = joblib.load("../Data/processed/X_train_processed.pkl")
X_test  = joblib.load("../Data/processed/X_test_processed.pkl")
y_train = pd.read_csv("../Data/splits/y_train.csv").squeeze()
y_test  = pd.read_csv("../Data/splits/y_test.csv").squeeze()

## Step 1: Baseline Model

Train a Random Forest with default settings and `class_weight='balanced'` to handle class imbalance. This gives us a score to beat.

In [3]:
rf_baseline = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_baseline.fit(X_train, y_train)

y_pred = rf_baseline.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High', 'Urgent']))

              precision    recall  f1-score   support

         Low       0.74      0.56      0.64        86
      Medium       0.65      0.82      0.72       146
        High       0.70      0.61      0.65        74
      Urgent       0.73      0.60      0.66        40

    accuracy                           0.68       346
   macro avg       0.70      0.65      0.67       346
weighted avg       0.69      0.68      0.68       346



In [4]:
## missing 40% of urgent cases - medium recall is strongest - low recall is weakest (struggles to identify low priority cases) 


#### Hyperparameter tuning 

In [5]:
# def objective(trial):
#     params = {
#         "n_estimators":      trial.suggest_int("n_estimators", 50, 100),
#         "max_depth":         trial.suggest_int("max_depth", 5, 15),
#         "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
#         "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
#         "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2"]),
#         "class_weight":      trial.suggest_categorical("class_weight", [None, "balanced"]),
#         "bootstrap":         trial.suggest_categorical("bootstrap", [True, False]),
#     }
#     model = clone(rf_baseline).set_params(**params)
#     cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
#     scores = cross_val_score(model, X_train, y_train, scoring="f1_macro", cv=cv, n_jobs=1)
#     return scores.mean()

# study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
# study.optimize(objective, n_trials=10)

# print(f"Best CV f1_macro: {round(study.best_value, 4)}")
# print(f"Best params: {study.best_params}")


### Feature Importance 

In [6]:
# preprocessor = joblib.load("../Data/processed/preprocessor.pkl")

# tfidf_features = preprocessor.named_transformers_["text"].get_feature_names_out()
# cat_features = preprocessor.named_transformers_["cat"].get_feature_names_out()
# tenure_features = preprocessor.named_transformers_["tenure"].get_feature_names_out()

# feature_names = list(tfidf_features) + list(cat_features) + list(tenure_features)

# importances = pd.Series(best_rf.feature_importances_, index=feature_names)
# top20 = importances.sort_values(ascending=False).head(20)

# plt.figure(figsize=(10, 6))
# top20.plot(kind="barh")
# plt.gca().invert_yaxis()
# plt.title("Top 20 Feature Importances — Random Forest")
# plt.tight_layout()
# plt.savefig("../Data/processed/top20_feature_importances.png", dpi=150)
# top20.to_csv("../Data/processed/top20_feature_importances.csv")
# plt.show()

 The model is driven almost entirely by structured fields (case type, plan tier, tenure, channel), the free text in case_summary is contributing almost nothing.

## Check wihtout case summary as a feature using baseline model

In [7]:
# cat_features = ["channel", "case_type", "category", "plan_tier", "day_of_week", "hour"]
# tenure_feature = ["customer_tenure_months"]

# preprocessor_no_text = ColumnTransformer(transformers=[
#     ("cat",    OneHotEncoder(handle_unknown="ignore"), cat_features),
#     ("tenure", KBinsDiscretizer(n_bins=5, encode="onehot", strategy="quantile"), tenure_feature),
# ])

# X_train_raw = pd.read_csv("../Data/splits/X_train.csv")
# X_test_raw  = pd.read_csv("../Data/splits/X_test.csv")

# X_train_raw["hour"] = pd.to_datetime(X_train_raw["created_at"]).dt.hour
# X_train_raw["day_of_week"] = pd.to_datetime(X_train_raw["created_at"]).dt.day_name()
# X_test_raw["hour"] = pd.to_datetime(X_test_raw["created_at"]).dt.hour
# X_test_raw["day_of_week"] = pd.to_datetime(X_test_raw["created_at"]).dt.day_name()

# X_train_raw = X_train_raw.drop(columns=["case_id", "created_at", "case_summary"])
# X_test_raw  = X_test_raw.drop(columns=["case_id", "created_at", "case_summary"])

# X_train_nt = preprocessor_no_text.fit_transform(X_train_raw)
# X_test_nt  = preprocessor_no_text.transform(X_test_raw)

# rf_no_text = RandomForestClassifier(class_weight="balanced", random_state=42)
# rf_no_text.fit(X_train_nt, y_train)

# y_pred_nt = rf_no_text.predict(X_test_nt)
# print(classification_report(y_test, y_pred_nt, target_names=["Low", "Medium", "High", "Urgent"]))


case_summary adds limited value as raw TF-IDF - it is best to drop it as it is too data heavey - it woudl be best to use this to extract structured signalas or 

## Step 3: Hyperparameter Tuning

Optuna searches for the best hyperparameters (number of trees, depth, etc.) using 50 trials.

To avoid crashing, tuning runs on structured features only (no TF-IDF). The best params are then applied to the full feature set in Step 4.

In [ ]:

# Slice off TF-IDF columns — tune on structured features only to avoid crashes
preprocessor = joblib.load("../Data/processed/preprocessor.pkl")
n_text_features = len(preprocessor.named_transformers_["text"].get_feature_names_out())
X_train_struct = X_train[:, n_text_features:]
X_test_struct  = X_test[:, n_text_features:]

print(f"Structured features only: {X_train_struct.shape[1]} columns (was {X_train.shape[1]})")

def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 500),
        "max_depth":         trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight":      trial.suggest_categorical("class_weight", [None, "balanced"]),
        "bootstrap":         trial.suggest_categorical("bootstrap", [True, False]),
    }
    model = clone(rf_baseline).set_params(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_struct, y_train, scoring="f1_macro", cv=cv, n_jobs=1)
    return scores.mean()

study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50)

print(f"Best CV f1_macro: {round(study.best_value, 4)}")
print(f"Best params: {study.best_params}")


Structured features only: 42 columns (was 4980)


[W 2026-03-03 19:25:46,858] Trial 28 failed with parameters: {'n_estimators': 320, 'max_depth': 14, 'min_samples_split': 14, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'class_weight': 'balanced', 'bootstrap': True} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/workspaces/Northstar-Desk-Decision-Support-Tool-/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_231296/2911994896.py", line 21, in objective
    scores = cross_val_score(model, X_train_struct, y_train, scoring="f1_macro", cv=cv, n_jobs=1)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/Northstar-Desk-Decision-Support-Tool-/.venv/lib/python3.12/site-packages/sklearn/utils/_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

## Step 4: Final Model

Take the best params from Step 3 and train one model on the full feature set (including TF-IDF). No cross-validation — just a single fit on all training data.

In [ ]:
# Train final model on FULL feature set (including TF-IDF) using best params from Optuna
final_model = RandomForestClassifier(**study.best_params, random_state=42)
final_model.fit(X_train, y_train)

y_pred_final = final_model.predict(X_test)
print(classification_report(y_test, y_pred_final, target_names=["Low", "Medium", "High", "Urgent"]))